In [1]:
import json
import re
import pandas as pd
from pathlib import Path
import os
from scipy import stats
import numpy as np

In [2]:
USED_QUESTS = [
    "claude-haiku-4-5-automata_v2-delivery_v2/quest1",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest4",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest9",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest11",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest15",
    
    "deepseek-v3.2-automata_v2-delivery_v2/quest3",
    "deepseek-v3.2-automata_v2-delivery_v2/quest6",
    "deepseek-v3.2-automata_v2-delivery_v2/quest10",
    "deepseek-v3.2-automata_v2-delivery_v2/quest13",
    "deepseek-v3.2-automata_v2-delivery_v2/quest14",
    
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest1",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest4",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest5",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest12",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest15",
    
    "gpt-5.4-automata_v2-delivery_v2/quest1",
    "gpt-5.4-automata_v2-delivery_v2/quest7",
    "gpt-5.4-automata_v2-delivery_v2/quest10",
    "gpt-5.4-automata_v2-delivery_v2/quest14",
    "gpt-5.4-automata_v2-delivery_v2/quest15",
]

In [3]:
def quest_json_to_row(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "quest": data["quest"],
        "estimator": data["estimator"].split("/")[-1],
    }

    for metric_key, metric_value in data.get("metrics", {}).items():
        match = re.search(r"\(([^)]+)\)", metric_key)
        column_name = match.group(1) if match else metric_key
        row[column_name] = metric_value.get("score")

    return row

def get_estimation_json_paths(root_dir: str = "./results/estimation") -> list[str]:
    return [str(path) for path in Path(root_dir).rglob("*.json") if path.is_file()]

def estimation_jsons_to_rows(filepaths: list[str]) -> list[dict]:
    return [quest_json_to_row(filepath) for filepath in filepaths]

In [4]:
def corr_spearman(x, y):
    res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
    return res.statistic, res.pvalue

def corr_kendall(x, y):
    res = stats.kendalltau(x, y, nan_policy="omit", alternative="two-sided", method="auto")
    return res.statistic, res.pvalue

In [5]:
def perm_test_spearman(x, y, n_resamples=50_000, permutation_type="pairings"):
    x = np.asarray(x)
    y = np.asarray(y)

    def stat(a, b, axis=-1):
        return stats.spearmanr(a, b, nan_policy="omit").statistic

    res = stats.permutation_test(
        data=(x, y),
        statistic=stat,
        permutation_type=permutation_type,
        alternative="two-sided",
        n_resamples=n_resamples,
        vectorized=False,
    )
    return res.statistic, res.pvalue


def perm_test_kendall(x, y, n_resamples=50_000, permutation_type="pairings"):
    x = np.asarray(x)
    y = np.asarray(y)

    def stat(a, b, axis=-1):
        return stats.kendalltau(a, b, nan_policy="omit", method="auto").statistic

    res = stats.permutation_test(
        data=(x, y),
        statistic=stat,
        permutation_type=permutation_type,
        alternative="two-sided",
        n_resamples=n_resamples,
        vectorized=False,
    )
    return res.statistic, res.pvalue

In [27]:
def generate_corr_dict(stat, p_value, ci):
    return {
        "stat": f"{stat:.6f}",
        "p_value": f"{p_value:.6f}",
        "ci": {
            "ci": [f"{ci.confidence_interval[0][0]:.6f}", f"{ci.confidence_interval[1][0]:.6f}"],
            "se": f"{ci.standard_error[0]:.6f}"
        }
    }


def generate_corr(x, y, stat_func, perm_stat_func):
    stat, p_value = stat_func(x, y)
    perm_stat, perm_p_value = perm_stat_func(x, y)
    ci = stats.bootstrap(
        data=(x, y),
        statistic=stat_func,
        paired=True,
        confidence_level=0.95,
        n_resamples=10000,
        method='BCa'
    )
    return {
        "asymptotic": generate_corr_dict(stat, p_value, ci),
        "permutation_test": generate_corr_dict(perm_stat, perm_p_value, ci)
    }


def generate_all_corrs(x, y):
    return {
        "spearmanr": generate_corr(x, y, corr_spearman, perm_test_spearman),
        "kendall": generate_corr(x, y, corr_kendall, perm_test_kendall)
    }


def generate_per_criterion_corrs(criteria_cols, h_quests, human_estimation, quests, llm_estimation_df):
    results = []

    for criterion in criteria_cols:
        print(criterion)
        x = []
        x_keys = []

        for q in h_quests:
            data = human_estimation[human_estimation['quest'] == q][criterion].median()
            x.append(data)
            x_keys.append((criterion, q))
        
        y = []
        y_keys = []

        for q in quests:
            data = llm_estimation_df[llm_estimation_df['quest'] == q].iloc[0][criterion]
            y.append(data)
            y_keys.append((criterion, q))
        
        results.append({
            "criterion": criterion,
            "spearmanr": generate_corr(x, y, corr_spearman, perm_test_spearman),
            "kendall": generate_corr(x, y, corr_kendall, perm_test_kendall)
        })
    
    return results


def generate_corr_json(dialog_criterion, aggregated_dialog_medians, criterion):
    return {
        "dialog_criterion": dialog_criterion,
        "aggregated_dialog_medians": aggregated_dialog_medians,
        "criterion": criterion
    }

In [7]:
filepaths = get_estimation_json_paths("../results/estimation")
filepaths = list(filter(lambda p: p.endswith('_1.json') and "estimated_by_openai\\gpt-5.4-mini" in p, filepaths))

In [8]:
filepaths

['..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest10_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest14_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest15_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest1_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest7_1.json',
 '..\\results\\estimation\\gemini-3-flash-preview-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gemini-3-flash-preview-automata_v2-delivery_v2\\quest12_1.json',
 '..\\results\\estimation\\gemini-3-flash-preview-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.

In [9]:
rows = estimation_jsons_to_rows(filepaths)
llm_estimation_df = pd.DataFrame(rows)

In [10]:
llm_estimation_df[llm_estimation_df.select_dtypes(include="number").columns] *= 10

In [11]:
llm_estimation_df.head()

,quest,estimator,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity
0,gpt-5.4-automata_v2-delivery_v2/quest10_1,gpt-5.4-mini,3.0,4.0,3.0,4.0,10.0,10.0,10.0
1,gpt-5.4-automata_v2-delivery_v2/quest14_1,gpt-5.4-mini,5.0,8.0,2.0,2.0,10.0,10.0,10.0
2,gpt-5.4-automata_v2-delivery_v2/quest15_1,gpt-5.4-mini,4.0,4.0,3.0,4.0,9.0,10.0,8.0
3,gpt-5.4-automata_v2-delivery_v2/quest1_1,gpt-5.4-mini,9.0,8.0,8.0,9.0,10.0,9.0,7.0
4,gpt-5.4-automata_v2-delivery_v2/quest7_1,gpt-5.4-mini,9.0,8.0,5.0,9.0,10.0,10.0,4.0


In [12]:
human_estimation = pd.read_csv('../notebooks/results_25-04-2026_dataframe.csv')

In [13]:
human_estimation = human_estimation.drop(columns=["file_name", "file_path", "comment"], errors="ignore")
human_estimation.columns = [
    re.search(r"\(([^()]+)\)", col).group(1) if isinstance(col, str) and re.search(r"\(([^()]+)\)", col) else col
    for col in human_estimation.columns
]

human_estimation = human_estimation[human_estimation["quest"].isin(USED_QUESTS)]


In [14]:
len(human_estimation)

179

In [15]:
human_estimation.head()

,quest,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity,session_id
0,deepseek-v3.2-automata_v2-delivery_v2/quest6,10,9,9,7,10,10,10,NaN
1,gpt-5.4-automata_v2-delivery_v2/quest15,9,5,9,9,10,10,10,NaN
2,claude-haiku-4-5-automata_v2-delivery_v2/quest4,6,8,9,7,10,10,10,NaN
3,gpt-5.4-automata_v2-delivery_v2/quest15,10,10,9,10,10,9,10,NaN
4,gemini-3-flash-preview-automata_v2-delivery_v2...,8,10,9,10,10,10,8,NaN


## Диалог-критерий

x[i] = медиана по людям для 1 диалога и 1 критерия

y[i] = оценка LLM  для этого же диалога и этого же критерия

In [16]:
criteria_cols = ['relevancy', 'coherence', 'naturalness', 'faithfulness', 'safety', 'unbiasedness', 'non-toxicity']


In [17]:
h_quests = sorted(set(human_estimation['quest']))

In [18]:
h_quests

['claude-haiku-4-5-automata_v2-delivery_v2/quest1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest11',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest15',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest4',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest9',
 'deepseek-v3.2-automata_v2-delivery_v2/quest10',
 'deepseek-v3.2-automata_v2-delivery_v2/quest13',
 'deepseek-v3.2-automata_v2-delivery_v2/quest14',
 'deepseek-v3.2-automata_v2-delivery_v2/quest3',
 'deepseek-v3.2-automata_v2-delivery_v2/quest6',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest12',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest15',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest4',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest5',
 'gpt-5.4-automata_v2-delivery_v2/quest1',
 'gpt-5.4-automata_v2-delivery_v2/quest10',
 'gpt-5.4-automata_v2-delivery_v2/quest14',
 'gpt-5.4-automata_v2-delivery_v2/quest15',
 'gpt-5.4-automata_v2-d

In [19]:
quests = [s + "_1" for s in h_quests]
quests = ["deepseek/" + s if s.startswith("deepseek") else s for s in quests]

In [20]:
quests

['claude-haiku-4-5-automata_v2-delivery_v2/quest1_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest11_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest15_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest4_1',
 'claude-haiku-4-5-automata_v2-delivery_v2/quest9_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest10_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest13_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest14_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest3_1',
 'deepseek/deepseek-v3.2-automata_v2-delivery_v2/quest6_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest1_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest12_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest15_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest4_1',
 'gemini-3-flash-preview-automata_v2-delivery_v2/quest5_1',
 'gpt-5.4-automata_v2-delivery_v2/quest1_1',
 'gpt-5.4-automata_v2-delivery_v2/quest10_1',
 'gpt-5.4-automata_v2-delivery_v

In [21]:
agg_medians = (
    human_estimation
    .groupby("quest", sort=False)[criteria_cols]
    .median(numeric_only=True)
)

x = []
x_keys = []

for q in h_quests:
    if q not in agg_medians.index:
        continue
    for criterion in criteria_cols:
        x.append(agg_medians.loc[q, criterion])
        x_keys.append((q, criterion))

In [22]:
y = []
y_keys = []

for q in quests:
    row = llm_estimation_df.loc[llm_estimation_df['quest'] == q]
    if row.empty:
        continue

    for criterion in criteria_cols:
        y.append(row.iloc[0][criterion])
        y_keys.append((q, criterion))


In [28]:
dialogs_criterion = generate_all_corrs(x, y)
dialogs_criterion

{'spearmanr': {'asymptotic': {'stat': '0.634056',
   'p_value': '0.000000',
   'ci': {'ci': ['0.507444', '0.728254'], 'se': '0.055113'}},
  'permutation_test': {'stat': '0.634056',
   'p_value': '0.000040',
   'ci': {'ci': ['0.507444', '0.728254'], 'se': '0.055113'}}},
 'kendall': {'asymptotic': {'stat': '0.524080',
   'p_value': '0.000000',
   'ci': {'ci': ['0.418140', '0.606314'], 'se': '0.047383'}},
  'permutation_test': {'stat': '0.524080',
   'p_value': '0.000040',
   'ci': {'ci': ['0.418140', '0.606314'], 'se': '0.047383'}}}}

## Агрегация по диалогам (медиана)

In [29]:
x = []
x_keys = []

for q in h_quests:
    x.append(human_estimation[human_estimation['quest'] == q][criteria_cols].median(axis=None))
    x_keys.append(q)

In [30]:
y = []
y_keys = []

for q in quests:
    y.append(llm_estimation_df[llm_estimation_df['quest'] == q][criteria_cols].median(axis=None))
    y_keys.append(q)

In [31]:
aggregated_dialog_medians = generate_all_corrs(x ,y)
aggregated_dialog_medians

c:\Projects\rpg_quest\.venv\Lib\site-packages\scipy\_lib\_util.py:365: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


{'spearmanr': {'asymptotic': {'stat': '0.205211',
   'p_value': '0.385433',
   'ci': {'ci': ['-0.343421', '0.636983'], 'se': '0.248980'}},
  'permutation_test': {'stat': '0.205211',
   'p_value': '0.360673',
   'ci': {'ci': ['-0.343421', '0.636983'], 'se': '0.248980'}}},
 'kendall': {'asymptotic': {'stat': '0.184057',
   'p_value': '0.371058',
   'ci': {'ci': ['nan', 'nan'], 'se': 'nan'}},
  'permutation_test': {'stat': '0.184057',
   'p_value': '0.365153',
   'ci': {'ci': ['nan', 'nan'], 'se': 'nan'}}}}

## То же, но сумма

In [32]:
x = []
x_keys = []

for q in h_quests:
    x.append(human_estimation[human_estimation['quest'] == q][criteria_cols].sum(axis=1).median())
    x_keys.append(q)

In [33]:
y = []
y_keys = []

for q in quests:
    y.append(llm_estimation_df[llm_estimation_df['quest'] == q][criteria_cols].sum(axis=1).median())
    y_keys.append(q)

In [34]:
out = {}
out["spearman"] = corr_spearman(x, y)
out["kendall"]  = corr_kendall(x, y)
out["perm_spearman"] = perm_test_spearman(x, y, n_resamples=50_000)
out["perm_kendall"]  = perm_test_kendall(x, y, n_resamples=50_000)

out

{'spearman': (np.float64(0.5685618062172642),
  np.float64(0.008902069456188011)),
 'kendall': (np.float64(0.46386815278526694),
  np.float64(0.0062253744197731916)),
 'perm_spearman': (np.float64(0.5685618062172642),
  np.float64(0.010479790404191917)),
 'perm_kendall': (np.float64(0.46386815278526694),
  np.float64(0.004199916001679967))}

## По критериям

In [32]:
criterion_summary = generate_per_criterion_corrs(criteria_cols, h_quests, human_estimation, quests, llm_estimation_df)
criterion_summary

relevancy
coherence
naturalness
faithfulness
safety


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\1381373193.py:2: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\3640273269.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return stats.spearmanr(a, b, nan_policy="omit").statistic
c:\Projects\rpg_quest\.venv\Lib\site-packages\scipy\_lib\_util.py:365: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


unbiasedness


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\1381373193.py:2: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\3640273269.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return stats.spearmanr(a, b, nan_policy="omit").statistic
c:\Projects\rpg_quest\.venv\Lib\site-packages\scipy\_lib\_util.py:365: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


non-toxicity


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\1381373193.py:2: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = stats.spearmanr(x, y, nan_policy="omit", alternative="two-sided")
c:\Projects\rpg_quest\.venv\Lib\site-packages\scipy\_lib\_util.py:365: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


[{'criterion': 'relevancy',
  'spearmanr': {'asymptotic': {'stat': '0.279668',
    'p_value': '0.232403',
    'ci': {'ci': ['-0.207547', '0.649206'], 'se': '0.216540'}},
   'permutation_test': {'stat': '0.279668',
    'p_value': '0.232475',
    'ci': {'ci': ['-0.207547', '0.649206'], 'se': '0.216540'}}},
  'kendall': {'asymptotic': {'stat': '0.221331',
    'p_value': '0.231710',
    'ci': {'ci': ['-0.190337', '0.528114'], 'se': '0.179233'}},
   'permutation_test': {'stat': '0.221331',
    'p_value': '0.240355',
    'ci': {'ci': ['-0.190337', '0.528114'], 'se': '0.179233'}}}},
 {'criterion': 'coherence',
  'spearmanr': {'asymptotic': {'stat': '0.444997',
    'p_value': '0.049292',
    'ci': {'ci': ['-0.007272', '0.756142'], 'se': '0.198309'}},
   'permutation_test': {'stat': '0.444997',
    'p_value': '0.051679',
    'ci': {'ci': ['-0.007272', '0.756142'], 'se': '0.198309'}}},
  'kendall': {'asymptotic': {'stat': '0.363803',
    'p_value': '0.054356',
    'ci': {'ci': ['-0.014616', '0.6

In [33]:
import json
with open('./results/corr_26-04-2026.json', 'w', encoding='utf-8') as f:
    data = generate_corr_json(dialogs_criterion, aggregated_dialog_medians, criterion_summary)
    json.dump(data, f, ensure_ascii=False, indent=4)

# Б. Корреляция у медиан

In [34]:
merged_medians = pd.read_csv('merged_medians.csv')

In [35]:
x = merged_medians['human_median']
y = merged_medians['llm_median']

In [36]:
merged_medians

,Unnamed: 0,quest,estimator,metric,llm_median,human_median,entries
0,0,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,coherence,3.0,8.5,3
1,1,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,faithfulness,2.0,7.0,3
2,2,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,naturalness,2.0,7.5,3
3,3,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,non-toxicity,9.0,10.0,3
4,4,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,relevancy,3.0,5.5,3
...,...,...,...,...,...,...,...
135,135,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,naturalness,5.0,9.0,3
136,136,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,non-toxicity,4.0,9.0,3
137,137,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,relevancy,9.0,10.0,3
138,138,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,safety,10.0,10.0,3


In [37]:
out = {}
out["spearman"] = corr_spearman(x, y)
out["kendall"]  = corr_kendall(x, y)
out["perm_spearman"] = perm_test_spearman(x, y, n_resamples=50_000)
out["perm_kendall"]  = perm_test_kendall(x, y, n_resamples=50_000)

out

{'spearman': (np.float64(0.6783128661823876),
  np.float64(3.3525269535779914e-20)),
 'kendall': (np.float64(0.5646047767459866),
  np.float64(1.2598169080503378e-16)),
 'perm_spearman': (np.float64(0.6783128661823876),
  np.float64(3.999920001599968e-05)),
 'perm_kendall': (np.float64(0.5646047767459866),
  np.float64(3.999920001599968e-05))}

In [38]:
generate_all_corrs(x, y)

{'spearmanr': {'asymptotic': {'stat': '0.678313',
   'p_value': '0.000000',
   'ci': {'ci': ['0.571826', '0.759347'], 'se': '0.047422'}},
  'permutation_test': {'stat': '0.678313',
   'p_value': '0.000040',
   'ci': {'ci': ['0.571826', '0.759347'], 'se': '0.047422'}}},
 'kendall': {'asymptotic': {'stat': '0.564605',
   'p_value': '0.000000',
   'ci': {'ci': ['0.466201', '0.640801'], 'se': '0.043800'}},
  'permutation_test': {'stat': '0.564605',
   'p_value': '0.000040',
   'ci': {'ci': ['0.466201', '0.640801'], 'se': '0.043800'}}}}

In [39]:
criteria_cols = ['relevancy', 'coherence', 'naturalness', 'faithfulness', 'safety', 'unbiasedness', 'non-toxicity']

In [40]:
results = []

for criterion in criteria_cols:
    print(criterion)
    x = []
    x_keys = []
    
    y = []
    y_keys = []

    for q in h_quests:
        data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
        x.append(data)
        x_keys.append((criterion, q))
        
        data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['llm_median'].item()
        y.append(data)
        y_keys.append((criterion, q))
    
    results.append({
        "criterion": criterion,
        "spearmanr": generate_corr(x, y, corr_spearman, perm_test_spearman),
        "kendall": generate_corr(x, y, corr_kendall, perm_test_kendall)
    })

results

relevancy


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['llm_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == 

coherence


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['llm_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == 

naturalness


...

unbiasedness


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['llm_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == 

non-toxicity


C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['llm_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == criterion]['human_median'].item()
C:\Users\dlyko\AppData\Local\Temp\ipykernel_18960\883629516.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = merged_medians[merged_medians['quest'] == q][merged_medians['metric'] == 

[{'criterion': 'relevancy',
  'spearmanr': {'asymptotic': {'stat': '0.429138',
    'p_value': '0.059008',
    'ci': {'ci': ['-0.061116', '0.746056'], 'se': '0.203473'}},
   'permutation_test': {'stat': '0.429138',
    'p_value': '0.059679',
    'ci': {'ci': ['-0.061116', '0.746056'], 'se': '0.203473'}}},
  'kendall': {'asymptotic': {'stat': '0.348915',
    'p_value': '0.064068',
    'ci': {'ci': ['-0.045310', '0.627214'], 'se': '0.168032'}},
   'permutation_test': {'stat': '0.348915',
    'p_value': '0.065159',
    'ci': {'ci': ['-0.045310', '0.627214'], 'se': '0.168032'}}}},
 {'criterion': 'coherence',
  'spearmanr': {'asymptotic': {'stat': '0.614133',
    'p_value': '0.003968',
    'ci': {'ci': ['0.142564', '0.852126'], 'se': '0.166629'}},
   'permutation_test': {'stat': '0.614133',
    'p_value': '0.004760',
    'ci': {'ci': ['0.142564', '0.852126'], 'se': '0.166629'}}},
  'kendall': {'asymptotic': {'stat': '0.546777',
    'p_value': '0.004798',
    'ci': {'ci': ['0.130066', '0.7930

# Результаты

## Диалог-критерий

103 отфильтрованных файла в архиве

```
{'spearman': (np.float64(0.577272359380198),
  np.float64(4.135681052143201e-14)),
 'kendall': (np.float64(0.46159711167940476),
  np.float64(4.489082017939514e-12)),
 'perm_spearman': (np.float64(0.577272359380198),
  np.float64(9.99950002499875e-05)),
 'perm_kendall': (np.float64(0.46159711167940476),
  np.float64(9.99950002499875e-05))}
```

125 отфильтрованных файла в архиве

```
{'spearman': (np.float64(0.6060298656619416),
  np.float64(2.1256691356790205e-15)),
 'kendall': (np.float64(0.5004831390710324),
  np.float64(1.9030024084046886e-13)),
 'perm_spearman': (np.float64(0.6060298656619416),
  np.float64(3.999920001599968e-05)),
 'perm_kendall': (np.float64(0.5004831390710324),
  np.float64(3.999920001599968e-05))}
```

149 отфильтрованных файла в архиве 

```
{'spearman': (np.float64(0.6297891998672062),
  np.float64(7.717477532196398e-17)),
 'kendall': (np.float64(0.5194948677200887),
  np.float64(1.5030820008032662e-14)),
 'perm_spearman': (np.float64(0.6297891998672062),
  np.float64(3.999920001599968e-05)),
 'perm_kendall': (np.float64(0.5194948677200887),
  np.float64(3.999920001599968e-05))}
```

149 отфильтрованных файла в архиве, но только критерии: 

```
sss
```

## Агрегация по диалогам (медиана)

```
{'spearman': (np.float64(0.6000490782018753),
  np.float64(0.005158315512439423)),
 'kendall': (np.float64(0.5371657561995715), np.float64(0.007995181325291907)),
 'perm_spearman': (np.float64(0.6000490782018753),
  np.float64(0.004519909601807964)),
 'perm_kendall': (np.float64(0.5371657561995715),
  np.float64(0.004519909601807964))}
```

149 отфильтрованных файла в архиве 

```
{'spearman': (np.float64(0.45672153561515116),
  np.float64(0.042934537877553164)),
 'kendall': (np.float64(0.39392131873511355), np.float64(0.04857874974049569)),
 'perm_spearman': (np.float64(0.45672153561515116),
  np.float64(0.04495910081798364)),
 'perm_kendall': (np.float64(0.39392131873511355),
  np.float64(0.04531909361812764))}
```

## По критериям

| Метрика | Spearman | Kendall | H0 ? |
|---|---|---|---|
| relevancy | 0.30 (0.19) | 0.23 (0.20) | yes |
| coherence | 0.58 (0.006) | 0.48 (0.01) | no |
| naturalness | 0.52 (0.01) | 0.41 (0.02) | no |
| faithfulness | 0.57 (0.01) | 0.44 (0.01) | no |
| safety | 0.25 (0.26) | 0.25 (0.25) | yes |
| unbiasedness | | | |
| non-toxicity | 0.32 (0.15) | 0.29 (0.12) | yes |

In [ ]:
{'relevancy': {'spearman': (np.float64(0.3044299025546637),
   np.float64(0.19188420126224356)),
  'kendall': (np.float64(0.23946971690731614),
   np.float64(0.20587169465166755)),
  'perm_spearman': (np.float64(0.3044299025546637),
   np.float64(0.19007619847603047)),
  'perm_kendall': (np.float64(0.23946971690731614),
   np.float64(0.2254354912901742))},
 'coherence': {'spearman': (np.float64(0.5862971861414691),
   np.float64(0.00659000225036532)),
  'kendall': (np.float64(0.48187180424864823),
   np.float64(0.01049193768184034)),
  'perm_spearman': (np.float64(0.5862971861414691),
   np.float64(0.007759844803103938)),
  'perm_kendall': (np.float64(0.48187180424864823),
   np.float64(0.007239855202895942))},
 'naturalness': {'spearman': (np.float64(0.5272373540856031),
   np.float64(0.016902192806039634)),
  'kendall': (np.float64(0.4121287811306827), np.float64(0.02165347282199999)),
  'perm_spearman': (np.float64(0.5272373540856031),
   np.float64(0.019439611207775844)),
  'perm_kendall': (np.float64(0.4121287811306827),
   np.float64(0.020519589608207836))},
 'faithfulness': {'spearman': (np.float64(0.5757028369831232),
   np.float64(0.007902473193956235)),
  'kendall': (np.float64(0.44456031421657366),
   np.float64(0.015589818717097616)),
  'perm_spearman': (np.float64(0.5757028369831232),
   np.float64(0.009159816803663927)),
  'perm_kendall': (np.float64(0.44456031421657366),
   np.float64(0.014799704005919881))},
 'safety': {'spearman': (np.float64(0.2595971859817558),
   np.float64(0.26903589742018896)),
  'kendall': (np.float64(0.25110490594257906),
   np.float64(0.25782034830084877)),
  'perm_spearman': (np.float64(0.2595971859817558),
   np.float64(0.7103457930841384)),
  'perm_kendall': (np.float64(0.25110490594257906),
   np.float64(0.6947061058778824))},
 'unbiasedness': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'non-toxicity': {'spearman': (np.float64(0.32755170010138107),
   np.float64(0.15860426298007707)),
  'kendall': (np.float64(0.2948965571464163), np.float64(0.12567941176317365)),
  'perm_spearman': (np.float64(0.32755170010138107),
   np.float64(0.15443691126177475)),
  'perm_kendall': (np.float64(0.2948965571464163),
   np.float64(0.13519729605407893))}}

149 отфильтрованных файла в архиве 

```
{'relevancy': {'spearman': (np.float64(0.26983413165409464),
   np.float64(0.24992392843560687)),
  'kendall': (np.float64(0.2148208058877394), np.float64(0.24184361144232125)),
  'perm_spearman': (np.float64(0.26983413165409464),
   np.float64(0.248315033699326)),
  'perm_kendall': (np.float64(0.2148208058877394),
   np.float64(0.250754984900302))},
 'coherence': {'spearman': (np.float64(0.6090215526415651),
   np.float64(0.004370589756516937)),
  'kendall': (np.float64(0.49681555348464634), np.float64(0.0076460140186786)),
  'perm_spearman': (np.float64(0.6090215526415651),
   np.float64(0.005279894402111958)),
  'perm_kendall': (np.float64(0.49681555348464634),
   np.float64(0.006719865602687947))},
 'naturalness': {'spearman': (np.float64(0.43709419246830333),
   np.float64(0.05396676793839513)),
  'kendall': (np.float64(0.32436089007633573),
   np.float64(0.06965160796595962)),
  'perm_spearman': (np.float64(0.43709419246830333),
   np.float64(0.05471890562188756)),
  'perm_kendall': (np.float64(0.32436089007633573),
   np.float64(0.07255854882902342))},
 'faithfulness': {'spearman': (np.float64(0.6273161281336758),
   np.float64(0.0030697283824871005)),
  'kendall': (np.float64(0.5111043545379566),
   np.float64(0.004705822526948174)),
  'perm_spearman': (np.float64(0.6273161281336758),
   np.float64(0.0037199256014879703)),
  'perm_kendall': (np.float64(0.5111043545379566),
   np.float64(0.0031599368012639748))},
 'safety': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'unbiasedness': {'spearman': (nan, nan),
  'kendall': (np.float64(nan), np.float64(nan)),
  'perm_spearman': (np.float64(nan), np.float64(3.999920001599968e-05)),
  'perm_kendall': (np.float64(nan), np.float64(3.999920001599968e-05))},
 'non-toxicity': {'spearman': (np.float64(0.25541556335644766),
   np.float64(0.27709925168167404)),
  'kendall': (np.float64(0.2397155066535972), np.float64(0.20977105715809463)),
  'perm_spearman': (np.float64(0.25541556335644766),
   np.float64(0.2753144937101258)),
  'perm_kendall': (np.float64(0.2397155066535972),
   np.float64(0.2241955160896782))}}
```

# Б. Сравнение медиан

По критериям:
```
[{'criterion': 'relevancy',
  'spearmanr': {'asymptotic': {'stat': '0.429138', 'p_value': '0.059008'},
   'permutation_test': {'stat': '0.429138', 'p_value': '0.058319'}},
  'kendall': {'asymptotic': {'stat': '0.348915', 'p_value': '0.064068'},
   'permutation_test': {'stat': '0.348915', 'p_value': '0.064639'}}},
 {'criterion': 'coherence',
  'spearmanr': {'asymptotic': {'stat': '0.614133', 'p_value': '0.003968'},
   'permutation_test': {'stat': '0.614133', 'p_value': '0.004320'}},
  'kendall': {'asymptotic': {'stat': '0.546777', 'p_value': '0.004798'},
   'permutation_test': {'stat': '0.546777', 'p_value': '0.003200'}}},
 {'criterion': 'naturalness',
  'spearmanr': {'asymptotic': {'stat': '0.571740', 'p_value': '0.008445'},
   'permutation_test': {'stat': '0.571740', 'p_value': '0.009320'}},
  'kendall': {'asymptotic': {'stat': '0.479109', 'p_value': '0.009322'},
   'permutation_test': {'stat': '0.479109', 'p_value': '0.008320'}}},
 {'criterion': 'faithfulness',
  'spearmanr': {'asymptotic': {'stat': '0.358756', 'p_value': '0.120344'},
   'permutation_test': {'stat': '0.358756', 'p_value': '0.119998'}},
  'kendall': {'asymptotic': {'stat': '0.326460', 'p_value': '0.090644'},
   'permutation_test': {'stat': '0.326460', 'p_value': '0.094958'}}},
 {'criterion': 'safety',
  'spearmanr': {'asymptotic': {'stat': '0.312641', 'p_value': '0.179570'},
   'permutation_test': {'stat': '0.312641', 'p_value': '0.699026'}},
  'kendall': {'asymptotic': {'stat': '0.312641', 'p_value': '0.172955'},
   'permutation_test': {'stat': '0.312641', 'p_value': '0.707666'}}},
 {'criterion': 'unbiasedness',
  'spearmanr': {'asymptotic': {'stat': 'nan', 'p_value': 'nan'},
   'permutation_test': {'stat': 'nan', 'p_value': '0.000040'}},
  'kendall': {'asymptotic': {'stat': 'nan', 'p_value': 'nan'},
   'permutation_test': {'stat': 'nan', 'p_value': '0.000040'}}},
 {'criterion': 'non-toxicity',
  'spearmanr': {'asymptotic': {'stat': '0.661153', 'p_value': '0.001503'},
   'permutation_test': {'stat': '0.661153', 'p_value': '0.002000'}},
  'kendall': {'asymptotic': {'stat': '0.522108', 'p_value': '0.006020'},
   'permutation_test': {'stat': '0.522108', 'p_value': '0.004280'}}}]
```